# Filtering

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.colors as mcolors
import numpy as np
import numpy.typing as npt
import xarray as xr

In [ ]:
from evaspa.common import filter

In [ ]:
# Create a discrete colormap with two colors
cmap = mcolors.ListedColormap(["red", "green"])
bounds = [-0.5, 0.5, 1.5]  # True=1, False=0
norm = mcolors.BoundaryNorm(bounds, cmap.N)

colors = [
    "#d73027",
    "#fc8d59",
    "#fee090",
    "#91bfdb",
    "#4575b4",
]  # choose any 5 colors
# Create colormap and normalization
lulc_cmap = mcolors.ListedColormap(colors)
bounds = [5, 15, 25, 35, 45, 55]  # boundaries between each value
lulc_norm = mcolors.BoundaryNorm(bounds, lulc_cmap.N)

## Detect valid pixels

### Inputs

In [ ]:
np.random.seed(0)
size = 100
lst = np.random.randint(low=290, high=300, size=(size, size))
clear = np.random.choice([0, 1], size=(size, size), p=[0.2, 0.8])
water = np.random.choice([0, 1], size=(size, size), p=[0.9, 0.1])
lulc = np.random.choice(
    [10, 20, 30, 40, 50], size=(size, size), p=[0.6, 0.1, 0.1, 0.1, 0.1]
)

In [ ]:
data = xr.Dataset(
    data_vars={
        "lst": (["y", "x"], lst),
        "clear": (["y", "x"], clear),
        "water": (["y", "x"], water),
        "lulc": (["y", "x"], lulc),
    },
    coords={
        "y": ("y", np.arange(size)),
        "x": ("x", np.arange(size)),
    },
    attrs={"description": "Test data"},
)

In [ ]:
data

In [ ]:
data["lst"].plot(cmap="Reds")

In [ ]:
data["clear"].plot(cmap=cmap, norm=norm)

In [ ]:
data["lulc"].plot(cmap=lulc_cmap, norm=lulc_norm)

### Simple case

In [ ]:
config = {
    "clear": {"op": "==", "value": 1},
}

In [ ]:
config

In [ ]:
filter.FilteringConfig.model_validate(config)

In [ ]:
valid = filter.detect_valid_pixels(data, config=config)

In [ ]:
valid.plot(cmap=cmap, norm=norm)

### More complex cases

In [ ]:
config1 = {
    "lst": {"and": [{"op": ">", "value": 292}, {"op": "<", "value": 297}]},
}

In [ ]:
config2 = {
    "lst": {
        "and": [
            {"op": ">", "value": "percentile(20)"},
            {"op": "<", "value": "percentile(80.5)"},
        ]
    },
}

In [ ]:
config3 = {
    "clear": {"op": "==", "value": 1},
    "water": {"op": "!=", "value": 1},
}

In [ ]:
config4 = {
    "lulc": {
        "or": [{"op": "==", "value": 10}, {"op": "==", "value": 20}],
    }
}

In [ ]:
config = config2

In [ ]:
config

In [ ]:
cfg = filter.FilteringConfig.model_validate(config).model_dump(by_alias=True)
cfg

In [ ]:
valid = filter.detect_valid_pixels(data, config)

In [ ]:
valid.plot(cmap=cmap, norm=norm)

### Combined case

In [ ]:
config = {
    "lst": {"and": [{"op": ">", "value": 291}, {"op": "<", "value": 298}]},
    "clear": {"op": "==", "value": 1},
    "water": {"op": "!=", "value": 1},
    "lulc": {
        "and": [{"op": "!=", "value": 40}, {"op": "!=", "value": 50}],
    },
}

In [ ]:
config

In [ ]:
filter.FilteringConfig.model_validate(config)

In [ ]:
valid = filter.detect_valid_pixels(data, config)

In [ ]:
valid.plot(cmap=cmap, norm=norm)

## Detect nan pixels

### Inputs

In [ ]:
# Generate a random mask
def random_mask(fraction: float) -> npt.NDArray:
    return np.random.rand(*lst.shape) < fraction

In [ ]:
np.random.seed(0)
size = 10
lst = np.random.randint(low=290, high=300, size=(size, size))
albedo = np.random.uniform(low=0.2, high=0.5, size=(size, size))
# Define the fraction of values to replace with NaN
fraction_to_replace = 0.1  # 30%
# Replace values with NaN
lst_with_nan = np.where(~random_mask(0.2), lst, np.nan)
albedo_with_nan = np.where(~random_mask(0.1), albedo, np.nan)

In [ ]:
data = xr.Dataset(
    data_vars={
        "lst": (["y", "x"], lst_with_nan),
        "albedo": (["y", "x"], albedo_with_nan),
    },
    coords={
        "y": ("y", np.arange(size)),
        "x": ("x", np.arange(size)),
    },
    attrs={"description": "Test data"},
)

In [ ]:
data["albedo"].plot()

In [ ]:
nan_mask = filter.detect_nan_pixels(data, ["lst"])
nan_mask.plot()

In [ ]:
nan_mask = filter.detect_nan_pixels(data, "all")
nan_mask.plot()